[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/04_Coverage_Gap_Analysis.ipynb)


# Notebook 04 — Coverage Gap Analysis
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Identify communities that fall outside walking distance of any functional borehole and quantify the population in those gaps.

**Requires:**
- Borehole dataset from the county water department (Excel file in `Kitui_WASHLAB/boreholes/`)
- Outputs from Notebook 02 in `Kitui_WASHLAB/outputs/`
- Ward boundaries in `Kitui_WASHLAB/boundaries/`

---
## What this notebook does

| Step | What it does |
|------|--------------|
| 1 | Loads and validates the borehole dataset |
| 2 | Creates service area buffers around functional boreholes |
| 3 | Calculates the coverage gap — areas outside any service area |
| 4 | Quantifies the population living in the gap using WorldPop |
| 5 | Aggregates results to ward level |
| 6 | Combines gap data with WASI scores to produce priority rankings |
| 7 | Exports outputs for the Streamlit app and report |

## Walking distance threshold

The Kenya government uses 1km as the standard walking distance threshold for basic water access in rural areas, consistent with national water policy. This notebook uses 1km as the primary threshold and also runs a 2km secondary analysis for comparison. Both results are exported.

> **Note:** This notebook requires the borehole dataset to run. All cells are fully built and ready. Once the dataset is placed in `Kitui_WASHLAB/boreholes/`, run all cells in order.

## Outputs
- `kitui_coverage_1km.geojson` — coverage polygons at 1km threshold
- `kitui_coverage_gap_1km.geojson` — gap polygons at 1km threshold
- `kitui_coverage_ward_table.csv` — ward-level coverage summary
- `kitui_priority_wards.csv` — wards ranked by combined WASI + gap score
- `kitui_coverage_map.png` — publication-ready map
- `kitui_boreholes.geojson` — validated borehole locations for Streamlit app


### 1. Setup

Installs required libraries and mounts Google Drive.
Sets the walking distance thresholds and coordinate reference systems used throughout.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas shapely rasterio rasterstats openpyxl matplotlib folium -q

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
import rasterstats
from shapely.geometry import Point, mapping
from shapely.ops import unary_union
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
BH    = DRIVE + 'boreholes/'

UTM_CRS  = 'EPSG:32637'  # UTM Zone 37N — Kenya, for distance calculations
WGS84    = 'EPSG:4326'   # Output CRS for maps and GeoJSON

# Walking distance thresholds
# 1km: Kenya national water policy standard for rural basic access
# 2km: Secondary threshold for extended service area analysis
WALK_1KM = 1000
WALK_2KM = 2000

print('Setup complete')
print(f'Primary threshold:   {WALK_1KM}m (1km — Kenya national standard)')
print(f'Secondary threshold: {WALK_2KM}m (2km — extended analysis)')


### 2. Load and Validate Borehole Data

Loads the borehole dataset from the county water department.
Validates coordinates, checks for missing values, and separates functional from non-functional boreholes.

Only functional boreholes are used for the service area and coverage gap calculations.
Non-functional boreholes are retained in the output for reference.

If the column names in the dataset differ from what is expected, the cell will print
all available columns so you can update the mapping at the top of the cell.


In [ ]:
# ── Load and validate borehole data ───────────────────────────────────────────
#
# Update these column name mappings to match the actual dataset
# Run the cell once to see available columns, then update below
COL_LAT        = 'Latitude'
COL_LON        = 'Longitude'
COL_FUNCTIONAL = 'Is_Functional'   # Expected: True/False or 1/0 or 'Yes'/'No'
COL_NAME       = 'Borehole_Name'
COL_WARD       = 'Ward'
SHEET_NAME     = 'B_Spatial_Analysis'
HEADER_ROW     = 2   # 0-indexed row number of the header

import os

# Find the borehole file
bh_files = [f for f in os.listdir(BH) if f.endswith(('.xlsx', '.xls', '.csv'))]
if not bh_files:
    raise FileNotFoundError(
        f'No borehole file found in {BH}\n'
        f'Place the Excel or CSV file in: Kitui_WASHLAB/boreholes/'
    )
bh_file = BH + bh_files[0]
print(f'Loading: {bh_file}')

# Load file
if bh_file.endswith('.csv'):
    df = pd.read_csv(bh_file)
else:
    try:
        df = pd.read_excel(bh_file, sheet_name=SHEET_NAME, header=HEADER_ROW)
    except Exception:
        # Try without sheet name
        df = pd.read_excel(bh_file, header=HEADER_ROW)

print(f'Rows loaded: {len(df)}')
print(f'Columns available: {df.columns.tolist()}')
print()

# Validate required columns exist
required = [COL_LAT, COL_LON, COL_FUNCTIONAL]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    print(f'MISSING COLUMNS: {missing_cols}')
    print('Update the column name mappings at the top of this cell and re-run.')
    raise ValueError(f'Required columns not found: {missing_cols}')

# Drop rows with missing coordinates
before = len(df)
df = df.dropna(subset=[COL_LAT, COL_LON])
print(f'Rows with valid coordinates: {len(df)} (dropped {before - len(df)} with missing coords)')

# Validate coordinate ranges for Kitui County
out_of_bounds = df[
    (df[COL_LON] < 37.5) | (df[COL_LON] > 39.2) |
    (df[COL_LAT] < -3.1) | (df[COL_LAT] > 0.0)
]
if len(out_of_bounds) > 0:
    print(f'WARNING: {len(out_of_bounds)} boreholes have coordinates outside Kitui County bounds')
    print(out_of_bounds[[COL_NAME, COL_LAT, COL_LON]].to_string())

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df[COL_LON], df[COL_LAT])],
    crs=WGS84
)

# Normalise functional status to boolean
func_vals = gdf[COL_FUNCTIONAL].unique()
print(f'Functional column unique values: {func_vals}')

if gdf[COL_FUNCTIONAL].dtype == bool:
    gdf['Functional'] = gdf[COL_FUNCTIONAL]
elif set(str(v).lower() for v in func_vals) <= {'yes', 'no', 'true', 'false', '1', '0', 'nan'}:
    gdf['Functional'] = gdf[COL_FUNCTIONAL].astype(str).str.lower().isin(['yes', 'true', '1'])
else:
    print('Could not auto-detect functional status. Set COL_FUNCTIONAL to the correct column.')
    gdf['Functional'] = False

functional    = gdf[gdf['Functional'] == True].copy()
nonfunctional = gdf[gdf['Functional'] == False].copy()

print()
print(f'Total GPS-verified boreholes: {len(gdf)}')
print(f'Functional (used for coverage): {len(functional)}')
print(f'Non-functional (reference only): {len(nonfunctional)}')


### 3. Build Service Areas

Creates circular buffer zones around each functional borehole at both the 1km and 2km thresholds.
Buffers are computed in UTM coordinates for accurate distance measurement.

All overlapping buffers are dissolved into a single coverage polygon representing the total area
within walking distance of at least one functional borehole.


In [ ]:
# ── Build service area buffers ─────────────────────────────────────────────────

functional_utm = functional.to_crs(UTM_CRS)

# 1km buffers
buffers_1km = functional_utm.geometry.buffer(WALK_1KM)
coverage_1km = unary_union(buffers_1km)
coverage_1km_gdf = gpd.GeoDataFrame(geometry=[coverage_1km], crs=UTM_CRS)

# 2km buffers
buffers_2km = functional_utm.geometry.buffer(WALK_2KM)
coverage_2km = unary_union(buffers_2km)
coverage_2km_gdf = gpd.GeoDataFrame(geometry=[coverage_2km], crs=UTM_CRS)

print(f'Service area coverage (1km): {coverage_1km.area / 1e6:.0f} km2')
print(f'Service area coverage (2km): {coverage_2km.area / 1e6:.0f} km2')
print(f'Kitui County area (approx):  30,496 km2')
print(f'County covered at 1km: {coverage_1km.area / 1e6 / 30496 * 100:.1f}%')
print(f'County covered at 2km: {coverage_2km.area / 1e6 / 30496 * 100:.1f}%')


### 4. Calculate Coverage Gap

The coverage gap is the area of Kitui County that falls outside all service areas.
It is calculated by subtracting the coverage polygon from the county boundary polygon.

Ward boundaries are loaded from Drive where Notebook 02 already saved them.
The gap polygon is intersected with ward boundaries to produce a ward-level breakdown.


In [ ]:
# ── Calculate coverage gap ─────────────────────────────────────────────────────

# Load ward boundaries (already on Drive from Notebook 02)
wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp').to_crs(WGS84)
if 'Ward' not in wards.columns:
    for col in ['NAME_3', 'WARD', 'ward', 'NAME']:
        if col in wards.columns:
            wards = wards.rename(columns={col: 'Ward'})
            break
print(f'Wards loaded: {len(wards)}')

# County boundary = union of all wards
county_boundary = wards.dissolve().to_crs(UTM_CRS)
county_geom_utm = county_boundary.geometry.iloc[0]

# Gap = county minus coverage
gap_1km = county_geom_utm.difference(coverage_1km)
gap_2km = county_geom_utm.difference(coverage_2km)
gap_1km_gdf = gpd.GeoDataFrame(geometry=[gap_1km], crs=UTM_CRS).to_crs(WGS84)
gap_2km_gdf = gpd.GeoDataFrame(geometry=[gap_2km], crs=UTM_CRS).to_crs(WGS84)

gap_pct_1km = gap_1km.area / county_geom_utm.area * 100
gap_pct_2km = gap_2km.area / county_geom_utm.area * 100

print(f'Coverage gap at 1km: {gap_pct_1km:.1f}% of county area')
print(f'Coverage gap at 2km: {gap_pct_2km:.1f}% of county area')

# Ward-level gap area
wards_utm = wards.to_crs(UTM_CRS)
wards_utm['Gap_Area_km2_1km'] = wards_utm.geometry.apply(
    lambda g: g.difference(coverage_1km).area / 1e6
)
wards_utm['Ward_Area_km2'] = wards_utm.geometry.area / 1e6
wards_utm['Gap_Pct_1km'] = (wards_utm['Gap_Area_km2_1km'] / wards_utm['Ward_Area_km2'] * 100).round(1)
wards = wards_utm.to_crs(WGS84)

print()
print('Top 10 wards by gap area (1km threshold):')
print(wards[['Ward', 'Ward_Area_km2', 'Gap_Area_km2_1km', 'Gap_Pct_1km']]
      .sort_values('Gap_Pct_1km', ascending=False)
      .head(10)
      .round(1)
      .to_string(index=False))


### 5. Population in Coverage Gap

Calculates the number of people living in the coverage gap at ward level.
Uses the WorldPop 2020 population raster saved by Notebook 02.

The population raster is masked to the gap polygon for each ward to count
how many people lack access to a functional borehole within walking distance.


In [ ]:
# ── Population in coverage gap ────────────────────────────────────────────────

POP_RASTER = OUT + 'kitui_wasi_c4_population.tif'

# Total population per ward
pop_total = rasterstats.zonal_stats(
    wards, POP_RASTER,
    stats=['sum'], nodata=float('nan')
)
wards['Pop_Total'] = [int(s['sum']) if s['sum'] else 0 for s in pop_total]

# Population in gap per ward (1km threshold)
# Intersect gap with each ward polygon
wards_utm = wards.to_crs(UTM_CRS)
gap_ward_geoms = []
for geom in wards_utm.geometry:
    gap_ward = geom.difference(coverage_1km)
    gap_ward_geoms.append(gap_ward)

gap_wards_gdf = gpd.GeoDataFrame(
    wards[['Ward']].copy(),
    geometry=gap_ward_geoms,
    crs=UTM_CRS
).to_crs(WGS84)

pop_gap = rasterstats.zonal_stats(
    gap_wards_gdf, POP_RASTER,
    stats=['sum'], nodata=float('nan')
)
wards['Pop_in_Gap_1km'] = [int(s['sum']) if s['sum'] else 0 for s in pop_gap]
wards['Pop_Covered_1km'] = wards['Pop_Total'] - wards['Pop_in_Gap_1km']
wards['Pop_Gap_Pct_1km'] = (
    wards['Pop_in_Gap_1km'] / wards['Pop_Total'].replace(0, np.nan) * 100
).round(1)

total_pop      = wards['Pop_Total'].sum()
total_gap_pop  = wards['Pop_in_Gap_1km'].sum()

print(f'Total county population (WorldPop 2020): {total_pop:,}')
print(f'Population in coverage gap (1km):        {total_gap_pop:,} ({total_gap_pop/total_pop*100:.1f}%)')
print(f'Population with coverage (1km):          {total_pop - total_gap_pop:,} ({(total_pop - total_gap_pop)/total_pop*100:.1f}%)')
print()
print('Top 10 wards by population in gap:')
print(wards[['Ward', 'Pop_Total', 'Pop_in_Gap_1km', 'Pop_Gap_Pct_1km']]
      .sort_values('Pop_in_Gap_1km', ascending=False)
      .head(10)
      .to_string(index=False))


### 6. Priority Ward Ranking

Combines the coverage gap data with the WASI scores from Notebook 02 to produce
a priority ranking for each ward.

The priority score combines three signals:
- WASI score (water stress — from Notebook 02)
- Population in gap (how many people lack coverage)
- Gap percentage (what share of the ward is uncovered)

Each signal is normalised to 0-1 and combined with equal weights.
Wards with high stress, large uncovered populations, and large gap areas rank highest.

This ranking feeds directly into Notebook 05 for final site prioritisation.


In [ ]:
# ── Priority ward ranking ──────────────────────────────────────────────────────

# Load WASI ward table from Notebook 02
wasi_table = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')
wards = wards.merge(wasi_table[['Ward', 'WASI_mean', 'Stress_Class']], on='Ward', how='left')

# Normalise each signal to 0-1
def norm(series):
    lo, hi = series.min(), series.max()
    if hi == lo: return series * 0
    return (series - lo) / (hi - lo)

wards['Score_WASI']    = norm(wards['WASI_mean'].fillna(0))
wards['Score_PopGap']  = norm(wards['Pop_in_Gap_1km'].fillna(0))
wards['Score_GapPct']  = norm(wards['Gap_Pct_1km'].fillna(0))

# Combined priority score — equal weights
wards['Priority_Score'] = (
    wards['Score_WASI'] * 0.40 +
    wards['Score_PopGap'] * 0.35 +
    wards['Score_GapPct'] * 0.25
).round(3)

wards['Priority_Rank'] = wards['Priority_Score'].rank(ascending=False).astype(int)

print('Top 15 priority wards for pilot site selection:')
print()
print(wards[['Priority_Rank', 'Ward', 'WASI_mean', 'Pop_in_Gap_1km',
             'Gap_Pct_1km', 'Priority_Score', 'Stress_Class']]
      .sort_values('Priority_Rank')
      .head(15)
      .to_string(index=False))


### 7. Export Outputs

Saves all outputs to `Kitui_WASHLAB/outputs/` for use in the Streamlit app and report.

Also exports a validated borehole GeoJSON for the interactive map.


In [ ]:
# ── Export outputs ─────────────────────────────────────────────────────────────

# Coverage polygons
coverage_1km_gdf.to_crs(WGS84).to_file(OUT + 'kitui_coverage_1km.geojson', driver='GeoJSON')
coverage_2km_gdf.to_crs(WGS84).to_file(OUT + 'kitui_coverage_2km.geojson', driver='GeoJSON')
print(f'Coverage GeoJSONs saved')

# Gap polygons
gap_1km_gdf.to_file(OUT + 'kitui_coverage_gap_1km.geojson', driver='GeoJSON')
gap_2km_gdf.to_file(OUT + 'kitui_coverage_gap_2km.geojson', driver='GeoJSON')
print(f'Gap GeoJSONs saved')

# Ward coverage table
csv_cols = ['Ward', 'WASI_mean', 'Stress_Class', 'Ward_Area_km2',
            'Gap_Area_km2_1km', 'Gap_Pct_1km', 'Pop_Total',
            'Pop_in_Gap_1km', 'Pop_Covered_1km', 'Pop_Gap_Pct_1km',
            'Priority_Score', 'Priority_Rank']
wards[csv_cols].sort_values('Priority_Rank').to_csv(
    OUT + 'kitui_coverage_ward_table.csv', index=False
)
print(f'Ward coverage table saved')

# Priority wards CSV
wards[csv_cols].sort_values('Priority_Rank').to_csv(
    OUT + 'kitui_priority_wards.csv', index=False
)
print(f'Priority wards table saved')

# Ward GeoJSON for Streamlit app
geojson_cols = ['Ward', 'WASI_mean', 'Stress_Class', 'Pop_Total',
                'Pop_in_Gap_1km', 'Gap_Pct_1km', 'Priority_Score',
                'Priority_Rank', 'geometry']
wards[geojson_cols].to_file(OUT + 'kitui_coverage_ward.geojson', driver='GeoJSON')
print(f'Ward GeoJSON saved')

# Borehole GeoJSON for Streamlit app
bh_cols = [c for c in [COL_NAME, COL_WARD, 'Functional', 'geometry'] if c in gdf.columns]
gdf[bh_cols].to_file(OUT + 'kitui_boreholes.geojson', driver='GeoJSON')
print(f'Borehole GeoJSON saved')

print()
print('All outputs saved to:', OUT)


### 8. Coverage Gap Map

Produces a two-panel map showing:
- Left: borehole locations and 1km service areas overlaid on ward boundaries
- Right: ward-level priority ranking combining WASI stress and coverage gap

The map is saved to Drive for inclusion in the Phase 1 report.


In [ ]:
# ── Coverage gap map ───────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

# Panel A: borehole coverage
ax = axes[0]
wards.plot(ax=ax, color='#F5F5F5', edgecolor='#AAAAAA', linewidth=0.5)
coverage_1km_gdf.to_crs(WGS84).plot(ax=ax, color='#2E75B6', alpha=0.25)
gap_1km_gdf.plot(ax=ax, color='#C00000', alpha=0.20)
functional.plot(ax=ax, color='#2E75B6', markersize=8, zorder=5, label='Functional borehole')
if len(nonfunctional) > 0:
    nonfunctional.plot(ax=ax, color='#888888', markersize=4, zorder=4,
                       marker='x', label='Non-functional')
ax.set_title('Borehole Coverage — 1km Walking Threshold\nBlue = covered | Red = gap', fontsize=10)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(fontsize=8)

# Panel B: priority ranking choropleth
ax = axes[1]
wards.plot(
    column='Priority_Score', cmap='RdYlGn_r',
    linewidth=0.5, edgecolor='white', legend=True,
    legend_kwds={'label': 'Priority score (higher = more urgent)', 'orientation': 'vertical'},
    ax=ax
)
# Label top 10 priority wards
for _, row in wards.nsmallest(10, 'Priority_Rank').iterrows():
    c = row.geometry.centroid
    ax.annotate(f"{row['Priority_Rank']}. {row['Ward']}",
                xy=(c.x, c.y), fontsize=5.5, ha='center', va='center',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.6, ec='none'))
ax.set_title('Ward Priority Ranking\nCombined WASI stress + coverage gap score', fontsize=10)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.suptitle('WASHLAB Pilot — Borehole Coverage Gap Analysis, Kitui County',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'kitui_coverage_map.png', dpi=200, bbox_inches='tight')
plt.show()
print('Coverage map saved')


### 9. Summary

Prints the full summary statistics for the report.
Run this cell last to confirm the analysis is complete.


In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────

print('COVERAGE GAP ANALYSIS SUMMARY')
print('=' * 55)
print(f'Total boreholes in dataset:      {len(gdf)}')
print(f'Functional boreholes:            {len(functional)}')
print(f'Non-functional boreholes:        {len(nonfunctional)}')
print()
print(f'Coverage at 1km threshold:       {100 - gap_pct_1km:.1f}% of county area')
print(f'Coverage gap at 1km:             {gap_pct_1km:.1f}% of county area')
print(f'Coverage at 2km threshold:       {100 - gap_pct_2km:.1f}% of county area')
print(f'Coverage gap at 2km:             {gap_pct_2km:.1f}% of county area')
print()
print(f'Total population (WorldPop 2020):{total_pop:,}')
print(f'Population in gap (1km):         {total_gap_pop:,} ({total_gap_pop/total_pop*100:.1f}%)')
print()
print('Top 5 priority wards:')
top5 = wards.nsmallest(5, 'Priority_Rank')[['Priority_Rank', 'Ward',
        'WASI_mean', 'Pop_in_Gap_1km', 'Priority_Score']]
print(top5.to_string(index=False))
print()
print('Outputs saved to Drive:')
for f in ['kitui_coverage_1km.geojson', 'kitui_coverage_gap_1km.geojson',
          'kitui_coverage_ward_table.csv', 'kitui_priority_wards.csv',
          'kitui_coverage_ward.geojson', 'kitui_boreholes.geojson',
          'kitui_coverage_map.png']:
    print(f'  {f}')
print()
print('Notebook 04 complete. Next: Notebook 05 — Site Prioritisation')
